# Homework Starter — Stage 05: Data Storage
Name: 
Date: 

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CHECKS = [(ROOT / ".env", "local configuration"), (ROOT / ".env.example", "safe template")]
print(f"Homework root: {ROOT}")
for path, note in CHECKS:
    print(f"[{'OK' if path.exists() else 'MISS'}] {path.name:<14} {note}")
assert all(path.exists() for path, _ in CHECKS)

Homework root: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework05
[OK] .env           local configuration
[OK] .env.example   safe template


In [3]:
import datetime as dt
import os

import pandas as pd
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
RAW = ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
PROC = ROOT / os.getenv("DATA_DIR_PROCESSED", "data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print("RAW ->", RAW)
print("PROC ->", PROC)

RAW -> /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework05/data/raw
PROC -> /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework05/data/processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np

rng = np.random.default_rng(5040)
dates = pd.date_range("2026-01-01", periods=20, freq="B")
df = pd.DataFrame(
    {
        "date": dates,
        "ticker": ["SPY"] * 20,
        "price": 680 + rng.normal(0, 1, 20).cumsum(),
        "volume": rng.integers(50_000_000, 100_000_000, 20),
    }
)
df.head()

,date,ticker,price,volume
0,2026-01-01,SPY,678.581719,54104086
1,2026-01-02,SPY,678.948098,99607193
2,2026-01-05,SPY,677.680257,81300682
3,2026-01-06,SPY,676.816020,75841010
4,2026-01-07,SPY,676.748043,59489226


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [5]:
from src.storage import write_df

RUN_STAMP = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = write_df(df, RAW / f"sample_{RUN_STAMP}.csv")
pq_path = write_df(df, PROC / f"sample_{RUN_STAMP}.parquet")
print("Saved CSV:", csv_path)
print("Saved Parquet:", pq_path)

Saved CSV: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework05/data/raw/sample_20260817-103816.csv
Saved Parquet: /Users/hansonsun/Documents/ChatGPT/FRE Bootcamp/homework/homework05/data/processed/sample_20260817-103816.parquet


## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [6]:
from pandas.testing import assert_frame_equal
from src.storage import read_df


def validate_loaded(original, reloaded):
    checks = {
        "shape_equal": original.shape == reloaded.shape,
        "columns_equal": original.columns.tolist() == reloaded.columns.tolist(),
        "date_is_datetime": pd.api.types.is_datetime64_any_dtype(reloaded["date"]),
        "price_is_numeric": pd.api.types.is_numeric_dtype(reloaded["price"]),
    }
    assert all(checks.values()), checks
    assert_frame_equal(original, reloaded, check_dtype=False, rtol=1e-10, atol=1e-12)
    return checks


df_csv = read_df(csv_path)
validate_loaded(df, df_csv)

{'shape_equal': True,
 'columns_equal': True,
 'date_is_datetime': True,
 'price_is_numeric': True}

In [7]:
df_pq = read_df(pq_path)
validate_loaded(df, df_pq)

{'shape_equal': True,
 'columns_equal': True,
 'date_is_datetime': True,
 'price_is_numeric': True}

## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [8]:
from src.storage import detect_format

print("CSV route:", detect_format(csv_path))
print("Parquet route:", detect_format(pq_path))
assert detect_format(csv_path) == "csv"
assert detect_format(pq_path) == 'parquet'

CSV route: csv
Parquet route: parquet


## 5) Documentation

The accompanying `README.md` documents the raw/processed folder split, CSV and Parquet tradeoffs, environment-driven paths, reusable suffix routing, and reload validation. The local `.env` contains the required `DATA_DIR_RAW` and `DATA_DIR_PROCESSED` values and remains ignored.

In [9]:
assert csv_path.is_file() and pq_path.is_file()
assert df_csv.shape == df_pq.shape == df.shape
assert df_csv.columns.tolist() == df_pq.columns.tolist() == df.columns.tolist()
assert np.isclose(df_csv.loc[0, 'price'], df_pq.loc[0, 'price'])
print('All Stage 05 checks passed.')

All Stage 05 checks passed.


## AI Assistance

AI assistance was used to implement and verify the storage workflow. Hanson Sun reviewed the outputs and documentation.